# Interactive Agent Session: Chapter 4 — HuggingFace smolagents Global Tax Auditor

**The Business Problem:** A global company processes transactions across UK, US, and EU. Each region has different VAT/tax rules. Manual auditing takes weeks and is error-prone.

**The Solution:** smolagents CodeAgent writes and executes Python code autonomously to compute taxes, compare against expected values, and flag discrepancies.

**Key Concept:** Unlike prompt-based agents, smolagents generates and runs **actual Python code** — making it deterministic and auditable.

In [ ]:
import sys
!{sys.executable} -m pip install --quiet smolagents python-dotenv

In [ ]:
%%capture smol_logs

import os
from dotenv import load_dotenv

load_dotenv()
USE_SIMULATION = not os.getenv("OPENAI_API_KEY")

# ── TRANSACTION LEDGER ──
TRANSACTIONS = [
    {"id": "TXN-001", "amount": 10000, "tax_rate": 0.20, "region": "UK", "reported_tax": 2000},
    {"id": "TXN-002", "amount": 5000, "tax_rate": 0.10, "region": "US-CA", "reported_tax": 500},
    {"id": "TXN-003", "amount": 8500, "tax_rate": 0.21, "region": "EU-DE", "reported_tax": 1700},  # ERROR: should be 1785
    {"id": "TXN-004", "amount": 3200, "tax_rate": 0.07, "region": "US-TX", "reported_tax": 224},
    {"id": "TXN-005", "amount": 15000, "tax_rate": 0.20, "region": "UK", "reported_tax": 2800},  # ERROR: should be 3000
]

# ── AUDIT LOGIC (what smolagents would generate) ──
audit_results = []
for txn in TRANSACTIONS:
    expected = round(txn["amount"] * txn["tax_rate"], 2)
    diff = round(txn["reported_tax"] - expected, 2)
    status = "✅ PASS" if abs(diff) < 0.01 else "❌ DISCREPANCY"
    audit_results.append({
        "id": txn["id"],
        "region": txn["region"],
        "amount": txn["amount"],
        "rate": f"{txn['tax_rate']*100:.0f}%",
        "expected": expected,
        "reported": txn["reported_tax"],
        "diff": diff,
        "status": status
    })

total_discrepancy = sum(r["diff"] for r in audit_results if "DISCREPANCY" in r["status"])

In [ ]:
from IPython.display import display, HTML

rows = ""
for r in audit_results:
    color = "#22c55e" if "PASS" in r["status"] else "#ef4444"
    rows += "<tr>"
    rows += '<td style="padding:10px; border-bottom:1px solid #222;">' + r["id"] + '</td>'
    rows += '<td style="padding:10px; border-bottom:1px solid #222;">' + r["region"] + '</td>'
    rows += '<td style="padding:10px; border-bottom:1px solid #222;">$' + f'{r["amount"]:,.0f}' + '</td>'
    rows += '<td style="padding:10px; border-bottom:1px solid #222;">' + r["rate"] + '</td>'
    rows += '<td style="padding:10px; border-bottom:1px solid #222;">$' + f'{r["expected"]:,.2f}' + '</td>'
    rows += '<td style="padding:10px; border-bottom:1px solid #222;">$' + f'{r["reported"]:,.2f}' + '</td>'
    rows += '<td style="padding:10px; border-bottom:1px solid #222; color:' + color + '; font-weight:700;">' + r["status"] + '</td>'
    rows += "</tr>"

html = '<style>@import url("https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;700&display=swap");</style>'
html += '<div style="max-width:950px; margin:30px auto; font-family:JetBrains Mono,monospace; background:#0a0a0a; padding:40px; border-radius:20px; border:3px solid #222; color:#fff;">'
html += '<div style="font-size:9px; color:#444; letter-spacing:4px; text-transform:uppercase; margin-bottom:20px;">SMOLAGENTS // CODE_AUDIT_ENGINE</div>'
html += '<table style="width:100%; border-collapse:collapse; font-size:12px;">'
html += '<tr style="color:#666; text-transform:uppercase; font-size:10px; letter-spacing:1px;"><th style="padding:10px; text-align:left;">ID</th><th style="padding:10px; text-align:left;">Region</th><th style="padding:10px; text-align:left;">Amount</th><th style="padding:10px; text-align:left;">Rate</th><th style="padding:10px; text-align:left;">Expected</th><th style="padding:10px; text-align:left;">Reported</th><th style="padding:10px; text-align:left;">Status</th></tr>'
html += rows
html += '</table>'
disc_color = "#ef4444" if total_discrepancy != 0 else "#22c55e"
html += '<div style="margin-top:20px; padding:15px; background:#111; border-radius:10px; border:1px solid #333;">'
html += '<span style="color:#666; font-size:11px;">TOTAL DISCREPANCY: </span>'
html += '<span style="color:' + disc_color + '; font-size:16px; font-weight:700;">$' + f'{total_discrepancy:,.2f}' + '</span>'
html += '</div></div>'
display(HTML(html))